In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from pyomo.environ import Objective, value, units as pyunits
from watertap.core.solvers import get_solver

# Import parameter sweep functions
from parameter_sweep import (
    LinearSample,
    parameter_sweep,
)

# Import existing flowsheet module
import Baseline_Flowsheet as baseline

solver = get_solver()

In [2]:
def build_and_solve():

    m = baseline.build()
    baseline.scale_system(m)
    baseline.add_costing(m)
    baseline.initialize_system(m)
    _ = baseline.solve_system(m)
    clear_output(wait=False)

    return m

In [3]:
m = build_and_solve()

In [6]:
# Save base case results
base_lcow = value(m.fs.costing.LCOW)
base_recovery = value(m.fs.RO.recovery_vol_phase[0, "Liq"])

base_A_comp = value(pyunits.convert(m.fs.RO.A_comp[0, "H2O"],to_units=pyunits.liter / pyunits.m**2 / pyunits.hour / pyunits.bar,))
base_mem_area = value(m.fs.RO.area)
base_pressure = value(pyunits.convert(m.fs.pump.control_volume.properties_out[0].pressure, to_units=pyunits.bar))

# Display base case results
print(f"\nBase LCOW: {base_lcow:.2f} $/m3")
print(f"Base Recovery: {base_recovery:.2f}")
print(f"\nBase A_comp: {base_A_comp:.2f} L/m2-hr-bar")
print(f"Base Membrane Area: {base_mem_area:.2f} m2")
print(f"Base Pump Outlet Pressure: {base_pressure:.2f} bar")

NameError: name 'm' is not defined

In [ ]:
# Parameter sweep

num_samples = 10
recoveries = np.linspace(0.1, 0.75, num_samples)


lcow_results_list = list()                                              # Create lists to store results
pressure_results_list = list()


m.fs.pump.control_volume.properties_out[0].pressure.unfix()             # Unfix variables to perform the sweep
m.fs.RO.length.fix()


def build_sweep_params(m, num_samples=10):

    sweep_params = {}
    sweep_params["Water Recovery"] = LinearSample(m.fs.RO.recovery_vol_phase, 0.1, 0.75, num_samples)
    return sweep_params



def build_outputs(m):
    """
    Create dictionary of outputs to record from the model.
    """
    outputs = {}

    outputs["Total Capital Cost"] = m.fs.costing.total_capital_cost
    outputs["Total Operating Cost"] = m.fs.costing.total_operating_cost
    outputs["LCOW"] = m.fs.costing.LCOW
    outputs["SEC"] = m.fs.costing.SEC

    # Unit-level outputs
    outputs["Pressure"] = m.fs.pump.control_volume.properties_out[0].pressure
    outputs["Recovery"] = m.fs.RO.recovery_vol_phase[0, "Liq"]
    outputs["Membrane Area"] = m.fs.RO.area
    outputs["Membrane Permeability"] = m.fs.RO.A_comp[0, "H2O"]

    # LCOW components
    outputs["LCOW Direct CAPEX Pump"] = m.fs.costing.LCOW_component_direct_capex["fs.pump"]
    outputs["LCOW Indirect CAPEX Pump"] = m.fs.costing.LCOW_component_indirect_capex["fs.pump"]
    outputs["LCOW Direct CAPEX ERD"] = m.fs.costing.LCOW_component_direct_capex["fs.erd"]
    outputs["LCOW Indirect CAPEX ERD"] = m.fs.costing.LCOW_component_indirect_capex["fs.erd"]
    outputs["LCOW Direct CAPEX RO"] = m.fs.costing.LCOW_component_direct_capex["fs.RO"]
    outputs["LCOW Indirect CAPEX RO"] = m.fs.costing.LCOW_component_indirect_capex["fs.RO"]
    outputs["LCOW Fixed OPEX Pump"] = m.fs.costing.LCOW_component_fixed_opex["fs.pump"]
    outputs["LCOW Fixed OPEX RO"] = m.fs.costing.LCOW_component_fixed_opex["fs.RO"]
    outputs["LCOW Fixed OPEX ERD"] = m.fs.costing.LCOW_component_fixed_opex["fs.erd"]
    outputs["LCOW Variable OPEX Electricity"] = (m.fs.costing.LCOW_aggregate_variable_opex["electricity"])
    return outputs


file_save = "one_parameter_sweep_results.csv"

results_array, results_dict = parameter_sweep(
    build_model=build_and_solve,
    build_sweep_params=build_sweep_params,
    build_sweep_params_kwargs={"num_samples": num_samples},
    build_outputs=build_outputs,
    csv_results_file_name=file_save,
)


from plot_functions import make_stacked_plot
make_stacked_plot(file_name="one_parameter_sweep_results.csv")